# UAV Fault Early-Warning Model — Clean Rebuild

This notebook consolidates every fix found during debugging:
- Correct 4-column fault detection (`engine_fault`, `aileron_fault`, `elevator_fault`, `rudder_fault`)
- A genuine forward-looking target (`fault_within_5s`), not current-state detection
- Train and test raw files combined into one pool, then **stratified by fault flights**
  into new train/val/test splits (fixes the old problem of test having almost no
  usable fault episodes)
- Class-balanced training (`scale_pos_weight`)
- Threshold chosen via cross-validation, not a single lucky/unlucky validation slice
- Clean, single final model + honest one-time test evaluation
- Artifacts saved for the Streamlit tester app


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, precision_recall_curve, auc, fbeta_score
)
from xgboost import XGBClassifier

pd.set_option("display.width", 120)


## 2. Load Raw Data

Edit the two paths below to point at your raw train and test CSV files.
Both files must contain: `bag`, `timestamp_ns`, sensor columns (`vel_x/y/z`,
`pitch/roll/yaw cmd/meas`, `imu_x/y/z`, `airspeed_*`), and the four raw fault
columns (`engine_fault`, `aileron_fault`, `elevator_fault`, `rudder_fault`).


In [4]:
TRAIN_RAW_PATH = "../data/train_raw.csv"   
TEST_RAW_PATH = "../data/test_raw.csv"     
train_raw = pd.read_csv(TRAIN_RAW_PATH)
test_raw = pd.read_csv(TEST_RAW_PATH)

print(f"Train raw: {train_raw.shape}")
print(f"Test raw:  {test_raw.shape}")


Train raw: (132937, 24)
Test raw:  (47875, 24)


## 3. Preprocessing Functions

Same logic used throughout the project, now consolidated into reusable functions.

In [ ]:
FAULT_COLS = ["engine_fault", "aileron_fault", "elevator_fault", "rudder_fault"]
FILL_COLS = [
    "vel_x", "vel_y", "vel_z", "pitch_cmd", "pitch_meas", "pitch_error",
    "yaw_cmd", "yaw_meas", "yaw_error"
]
GAP_NS_THRESHOLD = 1000 * 1e9  # 1000 seconds
HORIZON_SEC = 5.0


def add_flight_id(data: pd.DataFrame) -> pd.DataFrame:
    data = data.sort_values(by=["bag", "timestamp_ns"]).reset_index(drop=True)
    data["time_gap_ns"] = data.groupby("bag", sort=False)["timestamp_ns"].diff()
    bag_changed = data["bag"] != data["bag"].shift(1)
    large_gap = data["time_gap_ns"] > GAP_NS_THRESHOLD
    data["flight_id"] = (bag_changed | large_gap).cumsum() - 1
    data.drop(columns=["time_gap_ns"], inplace=True)
    return data


def add_flight_time(data: pd.DataFrame) -> pd.DataFrame:
    data["flight_time_sec"] = data.groupby("flight_id", sort=False)["timestamp_ns"].transform(
        lambda x: (x - x.min()) / 1e9
    )
    return data


def fill_missing(data: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in FILL_COLS if c in data.columns]
    data[cols] = data.groupby("flight_id", sort=False)[cols].ffill()
    return data


def add_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    data["yaw_error_clean"] = ((data["yaw_error"] + 180) % 360) - 180
    data["ground_speed"] = np.sqrt(data["vel_x"] ** 2 + data["vel_y"] ** 2 + data["vel_z"] ** 2)
    return data


def add_binary_fault(data: pd.DataFrame) -> pd.DataFrame:
    """binary_fault = 1 if ANY of the 4 raw actuator/engine fault columns is > 0."""
    present = [c for c in FAULT_COLS if c in data.columns]
    if not present:
        raise KeyError(f"None of {FAULT_COLS} found in the data.")
    data["binary_fault"] = (data[present] > 0).any(axis=1).astype(int)
    return data


def compute_fault_onset(data: pd.DataFrame) -> pd.DataFrame:
    """Marks the first frame of each fault episode (0 -> 1 transition)."""
    data = data.sort_values(["flight_id", "flight_time_sec"]).reset_index(drop=True)
    data["fault_onset"] = (
        (data["binary_fault"] == 1)
        & (data.groupby("flight_id")["binary_fault"].shift(1).fillna(0).eq(0))
    ).astype(int)
    return data


def add_future_fault_label(data: pd.DataFrame, horizon_sec: float = HORIZON_SEC) -> pd.DataFrame:
    """fault_within_5s = 1 if a fault_onset occurs within the next `horizon_sec`
    seconds, within the same flight."""
    data = data.sort_values(["flight_id", "flight_time_sec"]).reset_index(drop=True)
    labels = np.zeros(len(data), dtype=int)

    for flight_id, group in data.groupby("flight_id"):
        idx = group.index.values
        times = group["flight_time_sec"].values
        onset_times = np.sort(times[group["fault_onset"].values == 1])
        if len(onset_times) == 0:
            continue
        lo = np.searchsorted(onset_times, times, side="right")
        hi = np.searchsorted(onset_times, times + horizon_sec, side="right")
        labels[idx] = (hi > lo).astype(int)

    data["fault_within_5s"] = labels
    return data


def preprocess(data: pd.DataFrame) -> pd.DataFrame:
    """Full pipeline, in order, for one raw dataframe (one file)."""
    data = add_flight_id(data)
    data = add_flight_time(data)
    data = fill_missing(data)
    data = add_engineered_features(data)
    data = add_binary_fault(data)
    data = compute_fault_onset(data)
    data = add_future_fault_label(data)
    return data


FEATURES = [
    "airspeed_cmd", "airspeed_meas", "airspeed_error",
    "roll_cmd", "roll_meas", "roll_error",
    "pitch_cmd", "pitch_meas", "pitch_error",
    "yaw_cmd", "yaw_meas", "yaw_error_clean",
    "vel_x", "vel_y", "vel_z",
    "imu_x", "imu_y", "imu_z",
    "ground_speed",
]


In [19]:
print("XGBoost features:")
for i, col in enumerate(FEATURES, 1):
    print(f"{i:2}. {col}")

print("\nTarget:")
print("fault_within_5s")

XGBoost features:
 1. airspeed_cmd
 2. airspeed_meas
 3. airspeed_error
 4. roll_cmd
 5. roll_meas
 6. roll_error
 7. pitch_cmd
 8. pitch_meas
 9. pitch_error
10. yaw_cmd
11. yaw_meas
12. yaw_error_clean
13. vel_x
14. vel_y
15. vel_z
16. imu_x
17. imu_y
18. imu_z
19. ground_speed

Target:
fault_within_5s


## 4. Apply Preprocessing to Both Files

In [6]:
df_train_full = preprocess(train_raw.copy())
df_test_full = preprocess(test_raw.copy())

print(f"Train: {df_train_full['flight_id'].nunique()} flights, {len(df_train_full)} frames")
print(f"Test:  {df_test_full['flight_id'].nunique()} flights, {len(df_test_full)} frames")
print(f"Train binary_fault rate: {df_train_full['binary_fault'].mean():.4%}")
print(f"Test binary_fault rate:  {df_test_full['binary_fault'].mean():.4%}")


Train: 30 flights, 132937 frames
Test:  6 flights, 47875 frames
Train binary_fault rate: 14.8356%
Test binary_fault rate:  14.0783%


## 5. Combine Into One Pool

Both files were originally separate, with test having very few usable fault
episodes. We combine everything into one pool and re-split it fairly below,
so every split (train/val/test) gets a proportional share of the rare fault
episodes.


In [7]:
df_test_offset = df_test_full.copy()
flight_id_offset = df_train_full["flight_id"].max() + 1
df_test_offset["flight_id"] = df_test_offset["flight_id"] + flight_id_offset

df_all = pd.concat([df_train_full, df_test_offset], ignore_index=True)

# Predictive subset: exclude frames that are already mid-fault (we only want to
# predict BEFORE a fault, not detect one that's already happening)
df_all_pred = df_all[df_all["binary_fault"] == 0].copy()

print(f"Combined: {df_all['flight_id'].nunique()} flights, {len(df_all)} frames")
print(f"Predictive subset: {len(df_all_pred)} frames "
      f"({df_all_pred['fault_within_5s'].mean():.4%} positive)")


Combined: 36 flights, 180812 frames
Predictive subset: 154350 frames (2.2689% positive)


## 6. Stratified Flight-Level Split (60/20/20)

Faulty flights and normal flights are split *separately* into train/val/test,
so each split gets a fair, randomized share of the rare fault episodes —
rather than one split getting almost all of them (which is what happened
originally).


In [8]:
all_faulty_flights = sorted(df_all[df_all["fault_onset"] == 1]["flight_id"].unique())
all_flights = df_all_pred["flight_id"].unique()
non_faulty_flights = sorted(set(all_flights) - set(all_faulty_flights))

print(f"Total faulty flights: {len(all_faulty_flights)}")
print(f"Total non-faulty flights: {len(non_faulty_flights)}")

np.random.seed(42)

def split_list(lst, train_frac=0.6, val_frac=0.2):
    lst = list(lst)
    np.random.shuffle(lst)
    n = len(lst)
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    return lst[:n_train], lst[n_train:n_train + n_val], lst[n_train + n_val:]

faulty_train, faulty_val, faulty_test = split_list(all_faulty_flights)
normal_train, normal_val, normal_test = split_list(non_faulty_flights)

train_flights = faulty_train + normal_train
val_flights = faulty_val + normal_val
test_flights = faulty_test + normal_test

print(f"\nFaulty flights -> train:{len(faulty_train)}  val:{len(faulty_val)}  test:{len(faulty_test)}")
print(f"Normal flights -> train:{len(normal_train)}  val:{len(normal_val)}  test:{len(normal_test)}")

train_mask = df_all_pred["flight_id"].isin(train_flights)
val_mask = df_all_pred["flight_id"].isin(val_flights)
test_mask = df_all_pred["flight_id"].isin(test_flights)

X_train_raw = df_all_pred.loc[train_mask, FEATURES].copy()
X_val_raw = df_all_pred.loc[val_mask, FEATURES].copy()
X_test_raw = df_all_pred.loc[test_mask, FEATURES].copy()

y_train = df_all_pred.loc[train_mask, "fault_within_5s"].copy()
y_val = df_all_pred.loc[val_mask, "fault_within_5s"].copy()
y_test = df_all_pred.loc[test_mask, "fault_within_5s"].copy()

flight_id_train = df_all_pred.loc[train_mask, "flight_id"].copy()
flight_id_val = df_all_pred.loc[val_mask, "flight_id"].copy()
flight_id_test = df_all_pred.loc[test_mask, "flight_id"].copy()

print(f"\nTrain positive rate: {y_train.mean():.4%}  ({len(y_train)} frames)")
print(f"Val positive rate:   {y_val.mean():.4%}  ({len(y_val)} frames)")
print(f"Test positive rate:  {y_test.mean():.4%}  ({len(y_test)} frames)")


Total faulty flights: 29
Total non-faulty flights: 7

Faulty flights -> train:17  val:6  test:6
Normal flights -> train:4  val:1  test:2

Train positive rate: 2.4302%  (93201 frames)
Val positive rate:   3.5722%  (18700 frames)
Test positive rate:  1.3404%  (42449 frames)


In [20]:
print("Number of training flights:", len(train_flights))
print("Number of validation flights:", len(val_flights))

overlap = set(train_flights) & set(val_flights)

print("Overlapping flights:", overlap)
print("Number of overlapping flights:", len(overlap))

Number of training flights: 21
Number of validation flights: 7
Overlapping flights: set()
Number of overlapping flights: 0


## 7. Scale Features (fit on train only)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)


## 8. Leave-One-Fault-Flight-Out Cross-Validation (on train+val pool)

Before picking a final threshold, validate across *every* faulty flight in
train+val — not just whichever ones happen to be in one split — so the
threshold we settle on is stable rather than fit to a handful of lucky/unlucky
examples.


In [10]:
THRESH_GRID = np.linspace(0.10, 0.90, 33)
cv_pool_flights = list(train_flights) + list(val_flights)
cv_pool = df_all_pred[df_all_pred["flight_id"].isin(cv_pool_flights)]
cv_faulty_flights = [f for f in faulty_train + faulty_val]

fold_results = []
threshold_f2_matrix = []

for held_out in cv_faulty_flights:
    fold_train_flights = [f for f in cv_pool_flights if f != held_out]
    fold_train = cv_pool[cv_pool["flight_id"].isin(fold_train_flights)]
    fold_val = cv_pool[cv_pool["flight_id"] == held_out]

    if fold_val["fault_within_5s"].sum() == 0:
        continue

    fold_scaler = StandardScaler()
    X_ft = fold_scaler.fit_transform(fold_train[FEATURES])
    X_fv = fold_scaler.transform(fold_val[FEATURES])
    y_ft = fold_train["fault_within_5s"]
    y_fv = fold_val["fault_within_5s"]

    pw = (y_ft == 0).sum() / max((y_ft == 1).sum(), 1)
    fold_model = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        scale_pos_weight=pw, eval_metric="aucpr", random_state=42, n_jobs=-1
    )
    fold_model.fit(X_ft, y_ft)

    probs_raw = fold_model.predict_proba(X_fv)[:, 1]
    probs_smooth = pd.Series(probs_raw).rolling(window=10, min_periods=1).mean().values

    prec, rec, _ = precision_recall_curve(y_fv, probs_smooth)
    pr_auc_fold = auc(rec, prec)

    f2s = [fbeta_score(y_fv, (probs_smooth >= t).astype(int), beta=2.0, zero_division=0) for t in THRESH_GRID]
    threshold_f2_matrix.append(f2s)

    best_idx = int(np.argmax(f2s))
    fold_results.append({
        "held_out_flight": held_out,
        "n_val_frames": len(y_fv),
        "n_val_positive": int(y_fv.sum()),
        "pr_auc": pr_auc_fold,
        "best_thresh_this_fold": THRESH_GRID[best_idx],
        "best_f2_this_fold": f2s[best_idx],
    })

cv_results_df = pd.DataFrame(fold_results)
print(cv_results_df.to_string(index=False))
print(f"\nMean PR-AUC: {cv_results_df['pr_auc'].mean():.4f} (std {cv_results_df['pr_auc'].std():.4f})")
print(f"Mean best F2: {cv_results_df['best_f2_this_fold'].mean():.4f} (std {cv_results_df['best_f2_this_fold'].std():.4f})")

threshold_f2_matrix = np.array(threshold_f2_matrix)
avg_f2_per_thresh = threshold_f2_matrix.mean(axis=0)
STABLE_THRESH = THRESH_GRID[np.argmax(avg_f2_per_thresh)]
print(f"\nMost stable threshold across folds: {STABLE_THRESH:.2f} (avg F2 = {avg_f2_per_thresh.max():.4f})")


 held_out_flight  n_val_frames  n_val_positive   pr_auc  best_thresh_this_fold  best_f2_this_fold
              34          2102              96 0.025507                  0.100           0.000000
              20          5257             187 0.079014                  0.100           0.000000
              15          6269              94 0.340232                  0.850           0.643087
              26          2443              95 0.179135                  0.100           0.428082
               9          3170              91 0.020644                  0.100           0.000000
              10          1984              90 0.044332                  0.100           0.000000
              25          2354              93 0.084247                  0.100           0.282318
               0          5860             189 0.124126                  0.550           0.257417
              33          7060             282 0.188937                  0.100           0.351682
              16    

## 9. Train Final Model (all train+val, balanced weight)

In [11]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=pos_weight,
    eval_metric="aucpr",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)
model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    verbose=50,
)


scale_pos_weight: 40.15
[0]	validation_0-aucpr:0.11532
[50]	validation_0-aucpr:0.17732
[100]	validation_0-aucpr:0.18470
[150]	validation_0-aucpr:0.17674
[184]	validation_0-aucpr:0.18355


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)

## 10. Final Threshold — use the CV-stable value, sanity check on val

In [12]:
FINAL_THRESH = STABLE_THRESH

val_probs_raw = model.predict_proba(X_val_scaled)[:, 1]
val_df = pd.DataFrame({"flight_id": flight_id_val.values, "raw_probs": val_probs_raw})
val_probs_smooth = val_df.groupby("flight_id")["raw_probs"].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
).values

val_preds = (val_probs_smooth >= FINAL_THRESH).astype(int)
p, r, _ = precision_recall_curve(y_val, val_probs_smooth)
print(f"Using CV-selected threshold: {FINAL_THRESH:.2f}")
print(f"Val PR-AUC: {auc(r, p):.4f}")
print(f"Val F2: {fbeta_score(y_val, val_preds, beta=2.0, zero_division=0):.4f}\n")
print(classification_report(y_val, val_preds, target_names=["No fault in 5s (0)", "Fault within 5s (1)"]))


Using CV-selected threshold: 0.10
Val PR-AUC: 0.1983
Val F2: 0.3719

                     precision    recall  f1-score   support

 No fault in 5s (0)       0.99      0.79      0.88     18032
Fault within 5s (1)       0.12      0.77      0.21       668

           accuracy                           0.79     18700
          macro avg       0.56      0.78      0.54     18700
       weighted avg       0.96      0.79      0.86     18700



## 11. Final, One-Time Evaluation on Test

This is the single honest check — the model and threshold were chosen using
only train+val cross-validation. This is the first and only time this
particular model touches the test split.


In [13]:
test_probs_raw = model.predict_proba(X_test_scaled)[:, 1]
test_df = pd.DataFrame({"flight_id": flight_id_test.values, "raw_probs": test_probs_raw})
test_probs_smooth = test_df.groupby("flight_id")["raw_probs"].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
).values

test_preds = (test_probs_smooth >= FINAL_THRESH).astype(int)
p_t, r_t, _ = precision_recall_curve(y_test, test_probs_smooth)
pr_auc_test = auc(r_t, p_t)
f2_test = fbeta_score(y_test, test_preds, beta=2.0, zero_division=0)

print(f"=== FINAL MODEL on TEST set (threshold={FINAL_THRESH:.2f}) ===")
print(f"Test PR-AUC: {pr_auc_test:.4f}")
print(f"Test F2: {f2_test:.4f}\n")
print(classification_report(y_test, test_preds, target_names=["No fault in 5s (0)", "Fault within 5s (1)"]))


=== FINAL MODEL on TEST set (threshold=0.10) ===
Test PR-AUC: 0.0981
Test F2: 0.2779

                     precision    recall  f1-score   support

 No fault in 5s (0)       1.00      0.90      0.94     41880
Fault within 5s (1)       0.08      0.67      0.15       569

           accuracy                           0.90     42449
          macro avg       0.54      0.79      0.55     42449
       weighted avg       0.98      0.90      0.93     42449



## 12. Save Artifacts for the Streamlit App

In [ ]:
# joblib.dump(model, "xgboost_f2_v4.pkl")
# joblib.dump(model, "xgboost_baseline_v4.pkl")  # same final model for both slots
# joblib.dump(scaler, "scaler_v4.pkl")
# joblib.dump(FEATURES, "features_v4.pkl")
# joblib.dump(FINAL_THRESH, "best_thresh_v4.pkl")
# joblib.dump(HORIZON_SEC, "horizon_sec_v4.pkl")

# print("Saved: xgboost_baseline.pkl, xgboost_f2.pkl, scaler.pkl, features.pkl, "
#       "best_thresh.pkl, horizon_sec.pkl")


Saved: xgboost_baseline.pkl, xgboost_f2.pkl, scaler.pkl, features.pkl, best_thresh.pkl, horizon_sec.pkl


In [26]:
print("=" * 70)
print("STEP 4A — FLIGHT-WISE SPLIT VERIFICATION")
print("=" * 70)

print("\nFlight counts:")
print(f"Train flights: {len(train_flights)}")
print(f"Val flights:   {len(val_flights)}")
print(f"Test flights:  {len(test_flights)}")

print("\nFlight overlap:")

print(
    "Train ∩ Val :",
    set(train_flights) & set(val_flights)
)

print(
    "Train ∩ Test:",
    set(train_flights) & set(test_flights)
)

print(
    "Val ∩ Test  :",
    set(val_flights) & set(test_flights)
)

# ------------------------------------------------------------
# Check fault composition
# ------------------------------------------------------------

def count_faulty_flights(flights):
    return sum(
        flight_id in all_faulty_flights
        for flight_id in flights
    )

print("\nFault composition:")

print(
    f"Train: {count_faulty_flights(train_flights)} faulty / "
    f"{len(train_flights)} total"
)

print(
    f"Val:   {count_faulty_flights(val_flights)} faulty / "
    f"{len(val_flights)} total"
)

print(
    f"Test:  {count_faulty_flights(test_flights)} faulty / "
    f"{len(test_flights)} total"
)

# ------------------------------------------------------------
# Check that every flight belongs to exactly one split
# ------------------------------------------------------------

combined = (
    set(train_flights)
    | set(val_flights)
    | set(test_flights)
)

print("\nCoverage check:")

print(
    f"All flights accounted for: "
    f"{len(combined) == len(all_flights)}"
)

print(
    f"Missing flights: "
    f"{set(all_flights) - combined}"
)

STEP 4A — FLIGHT-WISE SPLIT VERIFICATION

Flight counts:
Train flights: 21
Val flights:   7
Test flights:  8

Flight overlap:
Train ∩ Val : set()
Train ∩ Test: set()
Val ∩ Test  : set()

Fault composition:
Train: 17 faulty / 21 total
Val:   6 faulty / 7 total
Test:  6 faulty / 8 total

Coverage check:
All flights accounted for: True
Missing flights: set()


In [28]:
# ============================================================
# STEP 4B DIAGNOSTIC — WHY ARE NO EVENTS FOUND?
# ============================================================

print("=" * 70)
print("STEP 4B DIAGNOSTIC")
print("=" * 70)

print("\nValidation flights:")
print(sorted(val_df["flight_id"].unique()))

print("\nFault onset counts by validation flight:")

onset_counts = (
    val_df
    .groupby("flight_id")["fault_onset"]
    .sum()
)

print(onset_counts)

print("\nTotal fault_onset rows in validation:")
print(val_df["fault_onset"].sum())

print("\nRows where fault_onset == 1:")
display(
    val_df[
        val_df["fault_onset"] == 1
    ][
        ["flight_id", "flight_time_sec", "fault_onset",
         "fault_within_5s"]
    ].head(30)
)

STEP 4B DIAGNOSTIC

Validation flights:
[3, 12, 22, 27, 28, 31, 35]

Fault onset counts by validation flight:
flight_id
3     0
12    0
22    0
27    0
28    0
31    0
35    0
Name: fault_onset, dtype: int32

Total fault_onset rows in validation:
0

Rows where fault_onset == 1:


,flight_id,flight_time_sec,fault_onset,fault_within_5s


In [29]:
# ============================================================
# STEP 4B — CORRECTED EVENT-LEVEL EVALUATION
# ============================================================

THRESHOLD = 0.10

print("=" * 70)
print("STEP 4B — EVENT-LEVEL EARLY WARNING")
print("=" * 70)


# ------------------------------------------------------------
# 1. Get validation predictions from df_all_pred
# ------------------------------------------------------------

val_pred = df_all_pred.loc[val_mask].copy()

val_pred = val_pred.sort_values(
    ["flight_id", "flight_time_sec"]
).reset_index(drop=True)

val_pred["fault_probability"] = model.predict_proba(
    val_pred[FEATURES]
)[:, 1]

val_pred["prediction"] = (
    val_pred["fault_probability"] >= THRESHOLD
).astype(int)


# ------------------------------------------------------------
# 2. Get TRUE fault onset events from df_all
# ------------------------------------------------------------

val_true = df_all[
    df_all["flight_id"].isin(val_flights)
].copy()

val_true = val_true.sort_values(
    ["flight_id", "flight_time_sec"]
)


# Find actual onset rows
onset_rows = val_true[
    val_true["fault_onset"] == 1
].copy()


print("\nValidation flights:")
print(sorted(val_flights))

print("\nActual onset rows in df_all:")
print(len(onset_rows))


# ------------------------------------------------------------
# 3. Remove t < 1 sec continuation events
# ------------------------------------------------------------

onset_rows = onset_rows[
    onset_rows["flight_time_sec"] >= 1.0
].copy()


events_df = onset_rows[
    ["flight_id", "flight_time_sec"]
].rename(
    columns={"flight_time_sec": "onset_time"}
).reset_index(drop=True)


print(
    f"Genuine validation fault events: {len(events_df)}"
)


# ------------------------------------------------------------
# 4. Evaluate each actual fault event
# ------------------------------------------------------------

results = []

for _, event in events_df.iterrows():

    flight_id = event["flight_id"]
    onset_time = event["onset_time"]

    # Predictions for this flight
    flight_pred = val_pred[
        val_pred["flight_id"] == flight_id
    ].sort_values("flight_time_sec")

    # Only predictions BEFORE the actual fault
    before_fault = flight_pred[
        flight_pred["flight_time_sec"] < onset_time
    ]

    # Alarms before fault
    alarms = before_fault[
        before_fault["prediction"] == 1
    ]

    if len(alarms) > 0:

        # Earliest alarm
        first_alarm = alarms.iloc[0]

        alarm_time = first_alarm["flight_time_sec"]

        warning_time = onset_time - alarm_time

        results.append({
            "flight_id": flight_id,
            "onset_time": onset_time,
            "detected": 1,
            "first_alarm_time": alarm_time,
            "warning_time_sec": warning_time,
            "max_probability_before_fault":
                before_fault["fault_probability"].max()
        })

    else:

        results.append({
            "flight_id": flight_id,
            "onset_time": onset_time,
            "detected": 0,
            "first_alarm_time": np.nan,
            "warning_time_sec": np.nan,
            "max_probability_before_fault":
                before_fault["fault_probability"].max()
        })


event_results = pd.DataFrame(results)


# ------------------------------------------------------------
# 5. Calculate event-level metrics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT-LEVEL RESULTS")
print("=" * 70)

if len(event_results) == 0:

    print("ERROR: No validation fault events were found.")

else:

    total_events = len(event_results)

    detected_events = int(
        event_results["detected"].sum()
    )

    missed_events = (
        total_events - detected_events
    )

    event_recall = (
        detected_events / total_events
    )

    print(
        f"Total genuine fault events : {total_events}"
    )

    print(
        f"Detected before fault      : {detected_events}"
    )

    print(
        f"Missed fault events        : {missed_events}"
    )

    print(
        f"Event-level recall         : "
        f"{event_recall:.4f}"
    )

    print(
        f"Event-level recall         : "
        f"{event_recall * 100:.2f}%"
    )


    # --------------------------------------------------------
    # Warning time
    # --------------------------------------------------------

    detected = event_results[
        event_results["detected"] == 1
    ]

    print("\n" + "=" * 70)
    print("WARNING TIME")
    print("=" * 70)

    if len(detected) > 0:

        print(
            f"Mean warning time   : "
            f"{detected['warning_time_sec'].mean():.3f} sec"
        )

        print(
            f"Median warning time : "
            f"{detected['warning_time_sec'].median():.3f} sec"
        )

        print(
            f"Minimum warning time: "
            f"{detected['warning_time_sec'].min():.3f} sec"
        )

        print(
            f"Maximum warning time: "
            f"{detected['warning_time_sec'].max():.3f} sec"
        )

    else:

        print("No faults were detected before onset.")


    # --------------------------------------------------------
    # Individual events
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("INDIVIDUAL FAULT EVENTS")
    print("=" * 70)

    display(
        event_results.sort_values(
            ["flight_id", "onset_time"]
        ).reset_index(drop=True)
    )

STEP 4B — EVENT-LEVEL EARLY WARNING

Validation flights:
[3, 12, 22, 27, 28, 31, 35]

Actual onset rows in df_all:
8
Genuine validation fault events: 7

EVENT-LEVEL RESULTS
Total genuine fault events : 7
Detected before fault      : 4
Missed fault events        : 3
Event-level recall         : 0.5714
Event-level recall         : 57.14%

WARNING TIME
Mean warning time   : 314.157 sec
Median warning time : 278.544 sec
Minimum warning time: 66.622 sec
Maximum warning time: 632.917 sec

INDIVIDUAL FAULT EVENTS


,flight_id,onset_time,detected,first_alarm_time,warning_time_sec,max_probability_before_fault
0,3.0,196.069189,0,NaN,NaN,0.072145
1,22.0,569.134332,1,80.174816,488.959516,0.104587
2,22.0,713.091892,1,80.174816,632.917077,0.104587
3,27.0,122.531777,0,NaN,NaN,0.063527
4,27.0,728.590486,1,660.461371,68.129115,0.107364
5,28.0,118.833552,0,NaN,NaN,0.092963
6,35.0,123.206626,1,56.584168,66.622458,0.107038


In [30]:
# ============================================================
# STEP 4C — STRICT 5-SECOND EVENT-LEVEL EVALUATION
# ============================================================

THRESHOLD = 0.10

results_5s = []

for _, event in events_df.iterrows():

    flight_id = event["flight_id"]
    onset_time = event["onset_time"]

    flight_pred = val_pred[
        val_pred["flight_id"] == flight_id
    ].sort_values("flight_time_sec")

    # ONLY the 5 seconds immediately before fault
    prediction_window = flight_pred[
        (flight_pred["flight_time_sec"] >= onset_time - 5.0) &
        (flight_pred["flight_time_sec"] < onset_time)
    ]

    alarms = prediction_window[
        prediction_window["prediction"] == 1
    ]

    if len(alarms) > 0:

        # Use the earliest alarm within the 5-sec window
        first_alarm = alarms.iloc[0]

        alarm_time = first_alarm["flight_time_sec"]

        warning_time = onset_time - alarm_time

        results_5s.append({
            "flight_id": flight_id,
            "onset_time": onset_time,
            "detected_within_5s": 1,
            "first_alarm_time": alarm_time,
            "warning_time_sec": warning_time,
            "max_probability_in_5s":
                prediction_window["fault_probability"].max()
        })

    else:

        results_5s.append({
            "flight_id": flight_id,
            "onset_time": onset_time,
            "detected_within_5s": 0,
            "first_alarm_time": np.nan,
            "warning_time_sec": np.nan,
            "max_probability_in_5s":
                prediction_window["fault_probability"].max()
        })


results_5s_df = pd.DataFrame(results_5s)


# ============================================================
# SUMMARY
# ============================================================

total_events = len(results_5s_df)

detected_events = int(
    results_5s_df["detected_within_5s"].sum()
)

missed_events = total_events - detected_events

event_recall_5s = detected_events / total_events


print("=" * 70)
print("STRICT 5-SECOND EVENT-LEVEL RESULTS")
print("=" * 70)

print(f"Total fault events       : {total_events}")
print(f"Detected within 5 sec    : {detected_events}")
print(f"Missed events            : {missed_events}")
print(f"Event recall             : {event_recall_5s:.4f}")
print(f"Event recall             : {event_recall_5s * 100:.2f}%")


# ============================================================
# WARNING TIME
# ============================================================

detected_5s = results_5s_df[
    results_5s_df["detected_within_5s"] == 1
]

if len(detected_5s) > 0:

    print("\n" + "=" * 70)
    print("WARNING TIME — WITHIN 5-SECOND WINDOW")
    print("=" * 70)

    print(
        f"Mean   : "
        f"{detected_5s['warning_time_sec'].mean():.3f} sec"
    )

    print(
        f"Median : "
        f"{detected_5s['warning_time_sec'].median():.3f} sec"
    )

    print(
        f"Min    : "
        f"{detected_5s['warning_time_sec'].min():.3f} sec"
    )

    print(
        f"Max    : "
        f"{detected_5s['warning_time_sec'].max():.3f} sec"
    )

else:

    print("\nNo events detected within the 5-second window.")


# ============================================================
# INDIVIDUAL EVENTS
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL 5-SECOND EVENTS")
print("=" * 70)

display(
    results_5s_df.sort_values(
        ["flight_id", "onset_time"]
    ).reset_index(drop=True)
)

STRICT 5-SECOND EVENT-LEVEL RESULTS
Total fault events       : 7
Detected within 5 sec    : 1
Missed events            : 6
Event recall             : 0.1429
Event recall             : 14.29%

WARNING TIME — WITHIN 5-SECOND WINDOW
Mean   : 1.504 sec
Median : 1.504 sec
Min    : 1.504 sec
Max    : 1.504 sec

INDIVIDUAL 5-SECOND EVENTS


,flight_id,onset_time,detected_within_5s,first_alarm_time,warning_time_sec,max_probability_in_5s
0,3.0,196.069189,0,NaN,NaN,0.031729
1,22.0,569.134332,0,NaN,NaN,0.002127
2,22.0,713.091892,0,NaN,NaN,0.001570
3,27.0,122.531777,0,NaN,NaN,0.029882
4,27.0,728.590486,1,727.08685,1.503636,0.107364
5,28.0,118.833552,0,NaN,NaN,0.004868
6,35.0,123.206626,0,NaN,NaN,0.021909


In [31]:
# ============================================================
# STEP 5 — INSPECT MODEL PROBABILITY BEFORE EACH FAULT
# ============================================================

print("=" * 70)
print("STEP 5 — PRE-FAULT PROBABILITY ANALYSIS")
print("=" * 70)

inspection = []

for _, event in events_df.iterrows():

    flight_id = event["flight_id"]
    onset_time = event["onset_time"]

    flight_pred = val_pred[
        val_pred["flight_id"] == flight_id
    ].sort_values("flight_time_sec")

    # --------------------------------------------------------
    # Windows
    # --------------------------------------------------------

    last_10s = flight_pred[
        (flight_pred["flight_time_sec"] >= onset_time - 10) &
        (flight_pred["flight_time_sec"] < onset_time)
    ]

    last_5s = flight_pred[
        (flight_pred["flight_time_sec"] >= onset_time - 5) &
        (flight_pred["flight_time_sec"] < onset_time)
    ]

    last_2s = flight_pred[
        (flight_pred["flight_time_sec"] >= onset_time - 2) &
        (flight_pred["flight_time_sec"] < onset_time)
    ]

    last_1s = flight_pred[
        (flight_pred["flight_time_sec"] >= onset_time - 1) &
        (flight_pred["flight_time_sec"] < onset_time)
    ]

    # --------------------------------------------------------
    # Closest probability to specific times
    # --------------------------------------------------------

    def probability_near(seconds_before):

        target_time = onset_time - seconds_before

        if len(flight_pred) == 0:
            return np.nan

        idx = (
            flight_pred["flight_time_sec"] - target_time
        ).abs().idxmin()

        return flight_pred.loc[
            idx, "fault_probability"
        ]

    inspection.append({

        "flight_id": flight_id,

        "onset_time": onset_time,

        "max_prob_10s":
            last_10s["fault_probability"].max(),

        "max_prob_5s":
            last_5s["fault_probability"].max(),

        "max_prob_2s":
            last_2s["fault_probability"].max(),

        "max_prob_1s":
            last_1s["fault_probability"].max(),

        "prob_at_5s":
            probability_near(5),

        "prob_at_2s":
            probability_near(2),

        "prob_at_1s":
            probability_near(1),

        "frames_in_5s":
            len(last_5s),

        "frames_in_10s":
            len(last_10s)
    })


inspection_df = pd.DataFrame(inspection)


display(
    inspection_df.sort_values(
        ["flight_id", "onset_time"]
    ).reset_index(drop=True)
)

STEP 5 — PRE-FAULT PROBABILITY ANALYSIS


,flight_id,onset_time,max_prob_10s,max_prob_5s,max_prob_2s,max_prob_1s,prob_at_5s,prob_at_2s,prob_at_1s,frames_in_5s,frames_in_10s
0,3.0,196.069189,0.031729,0.031729,0.010010,0.010010,0.002073,0.007444,0.004529,90,183
1,22.0,569.134332,0.011704,0.002127,0.000567,0.000567,0.001367,0.000507,0.000507,97,195
2,22.0,713.091892,0.001637,0.001570,0.001570,0.001037,0.000615,0.001068,0.000719,100,195
3,27.0,122.531777,0.029882,0.029882,0.008435,0.004606,0.000546,0.008435,0.003374,94,185
4,27.0,728.590486,0.107364,0.107364,0.107364,0.007925,0.014132,0.005827,0.012521,98,195
5,28.0,118.833552,0.092963,0.004868,0.002002,0.001029,0.003303,0.002002,0.001029,97,194
6,35.0,123.206626,0.021909,0.021909,0.019320,0.012073,0.004247,0.019320,0.010057,92,189


In [32]:
# ============================================================
# STEP 6 — PRE-FAULT FEATURE BEHAVIOR
# ============================================================

print("=" * 70)
print("STEP 6 — PRE-FAULT FEATURE ANALYSIS")
print("=" * 70)

# Use validation data
val_features_df = df_all.loc[
    df_all["flight_id"].isin(val_flights)
].copy()

feature_stats = []

for _, event in events_df.iterrows():

    flight_id = event["flight_id"]
    onset_time = event["onset_time"]

    flight = val_features_df[
        val_features_df["flight_id"] == flight_id
    ].sort_values("flight_time_sec")

    # 5 sec immediately before fault
    pre_fault = flight[
        (flight["flight_time_sec"] >= onset_time - 5) &
        (flight["flight_time_sec"] < onset_time)
    ]

    # 20 sec baseline before that
    baseline = flight[
        (flight["flight_time_sec"] >= onset_time - 25) &
        (flight["flight_time_sec"] < onset_time - 5)
    ]

    for feature in FEATURES:

        pre_mean = pre_fault[feature].mean()
        base_mean = baseline[feature].mean()

        pre_std = pre_fault[feature].std()
        base_std = baseline[feature].std()

        # Standardized difference
        pooled_std = np.sqrt(
            (pre_std ** 2 + base_std ** 2) / 2
        )

        if pooled_std > 1e-12:
            effect = abs(pre_mean - base_mean) / pooled_std
        else:
            effect = 0.0

        feature_stats.append({
            "flight_id": flight_id,
            "onset_time": onset_time,
            "feature": feature,
            "baseline_mean": base_mean,
            "pre_fault_mean": pre_mean,
            "baseline_std": base_std,
            "pre_fault_std": pre_std,
            "effect_size": effect
        })


feature_stats_df = pd.DataFrame(feature_stats)


# Average effect across fault events
feature_summary = (
    feature_stats_df
    .groupby("feature")
    .agg(
        mean_effect_size=("effect_size", "mean"),
        median_effect_size=("effect_size", "median"),
        max_effect_size=("effect_size", "max")
    )
    .sort_values(
        "mean_effect_size",
        ascending=False
    )
)


print("\nTop features showing change before faults:")

display(feature_summary)

STEP 6 — PRE-FAULT FEATURE ANALYSIS

Top features showing change before faults:


,mean_effect_size,median_effect_size,max_effect_size
feature,,,
vel_y,2.237805,1.731373,4.752630
roll_cmd,1.708670,1.587897,4.708539
roll_meas,1.707802,1.205402,3.715469
ground_speed,1.621355,0.854233,5.309058
yaw_cmd,1.569285,1.383195,3.145983
imu_x,1.439628,1.105157,3.723540
yaw_error_clean,1.302263,0.665708,4.573666
vel_x,1.204921,1.489513,2.429563
yaw_meas,1.060178,0.612683,2.519997
